# catboosting_v2.ipynb — 仅 CatBoost（使用 v2 流水特征）
保持 v1 骨架，不引入 LGBM/搜索；GPU 兼容；严格避免泄漏。

In [1]:

# 配置
REF_DATE_STR = "2025-08-31"

TRAIN_CSV = "train.csv"
TESTAA_CSV = "testaa.csv"
TESTAB_CSV = "testab.csv"

TRAIN_STM_FEAT = "train_statement_feature_v2.csv"   # 优先用 v2
TESTAA_STM_FEAT = "testaa_statement_feature_v2.csv"
TESTAB_STM_FEAT = "testab_statement_feature_v2.csv"

# 若 v2 不存在，主程序内会回退到默认名 *_statement_feature.csv

OUT_DIR = "outputs_cat_v2"
N_FOLDS = 5
RANDOM_STATE = 42
USE_GPU = True


In [2]:
# CatBoost 参数（从旧 catboosting.ipynb 解析或安全默认）
CAT_PARAMS = {
  "iterations": 5000,
  "learning_rate": 0.03,
  "depth": 6,
  "l2_leaf_reg": 9.0,
  "bootstrap_type": "Bernoulli",
  "subsample": 0.75,
  "random_strength": 1.5,
  "one_hot_max_size": 10,
  "loss_function": "Logloss",
  "eval_metric": "AUC",
  "verbose": 200
}

In [3]:

import os, json, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

os.makedirs(OUT_DIR, exist_ok=True)
REF_DATE = pd.Timestamp(REF_DATE_STR)

def _to_days_since_now(unix_series):
    dt = pd.to_datetime(unix_series, unit='s', utc=True, errors='coerce').dt.tz_convert(None)
    return (REF_DATE - dt).dt.days

def prepare_main_table(df):
    use_cols = [
        'id','title','career','zip_code','residence','loan','term','interest_rate',
        'issue_time','syndicated','installment','record_time','history_time',
        'total_accounts','balance_accounts','balance_limit','balance','level'
    ] + (['label'] if 'label' in df.columns else [])
    df = df[use_cols].copy()
    for c in ['issue_time','record_time','history_time']:
        df[f'{c}_days'] = _to_days_since_now(df[c])
    df['diff_issue_record_days']   = df['issue_time_days'] - df['record_time_days']
    df['diff_issue_history_days']  = df['issue_time_days'] - df['history_time_days']
    df['diff_record_history_days'] = df['record_time_days'] - df['history_time_days']
    df['utilization']    = (df['balance'] / df['balance_limit']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 10)
    df['accounts_ratio'] = (df['balance_accounts'] / df['total_accounts']).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(0, 1)
    def split_level(x):
        if isinstance(x, str) and len(x) >= 2: return x[0], x[1:]
        return 'NA', 'NA'
    lv = df['level'].fillna('NA')
    df['grade'], df['subgrade'] = zip(*lv.map(split_level))
    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    for c in cat_cols:
        df[c] = df[c].astype(str).fillna('NA')
    return df

def merge_statement_feats(main_df, v2_path, fallback_path):
    # 优先 v2；不存在则 fallback
    path = v2_path if os.path.exists(v2_path) else fallback_path
    if not os.path.exists(path):
        main_df['has_stm'] = 0
        return main_df
    stm = pd.read_csv(path)
    if 'label' in stm.columns: stm = stm.drop(columns=['label'])
    out = main_df.merge(stm, on='id', how='left')
    stm_cols = [c for c in stm.columns if c!='id']
    out['has_stm'] = (out[stm_cols].notna().any(axis=1)).astype(int)
    out[stm_cols] = out[stm_cols].fillna(0)
    return out

def get_feature_lists(df):
    drop_cols = ['id','label']
    features = [c for c in df.columns if c not in drop_cols]
    cat_cols = ['title','career','zip_code','residence','term','syndicated','installment','level','grade','subgrade']
    cat_cols = [c for c in cat_cols if c in features]
    return features, cat_cols

def sanitize_cat_params(params, use_gpu=False, y=None, seed=1337):
    p = dict(params)
    if use_gpu and 'rsm' in p:
        p.pop('rsm', None)  # GPU 不支持 rsm（仅 pairwise）
    if p.get('bootstrap_type','').lower()=='bayesian' and 'subsample' in p:
        p.pop('subsample', None); p.setdefault('bagging_temperature', 1.0)
    if use_gpu:
        p['task_type'] = 'GPU'
    if y is not None:
        pos, neg = int(np.sum(y)), int(len(y)-np.sum(y))
        p['scale_pos_weight'] = float(neg/max(pos,1))
    p.setdefault('loss_function','Logloss')
    p.setdefault('eval_metric','AUC')
    p.setdefault('verbose', False)
    p['random_seed'] = seed
    return p

def train_catboost_cv(df_train, features, cat_cols, params, n_folds=5, seed=42):
    X = df_train[features]
    y = df_train['label'].astype(int).values
    if len(y)==0:
        raise RuntimeError("训练集 label 为空")
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]
    p = sanitize_cat_params(params, use_gpu=USE_GPU, y=y, seed=seed)
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof = np.zeros(len(y), dtype=float)
    fold_scores, models = [], []
    for f,(tr_idx,va_idx) in enumerate(skf.split(X, y),1):
        train_pool = Pool(X.iloc[tr_idx], label=y[tr_idx], cat_features=cat_idx)
        valid_pool = Pool(X.iloc[va_idx], label=y[va_idx], cat_features=cat_idx)
        model = CatBoostClassifier(**p)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True, early_stopping_rounds=300)
        pred = model.predict_proba(valid_pool)[:,1]
        oof[va_idx] = pred
        fold_scores.append(roc_auc_score(y[va_idx], pred))
        models.append(model)
    mean_auc = roc_auc_score(y, oof)
    return mean_auc, fold_scores, oof, models


In [4]:

# 读取训练与合并 v2 流水
tr = pd.read_csv(TRAIN_CSV)
tr = prepare_main_table(tr)
tr = merge_statement_feats(tr, TRAIN_STM_FEAT, "train_statement_feature.csv")

features, cat_cols = get_feature_lists(tr)
print("Train shape:", tr.shape, "Features:", len(features), "Cat cols:", len(cat_cols))
print("Label dist:", tr['label'].value_counts(normalize=True).round(4).to_dict())


Train shape: (53480, 46) Features: 44 Cat cols: 10
Label dist: {0: 0.8151, 1: 0.1849}


In [5]:

# 训练 + OOF
mean_auc, fold_scores, oof, models = train_catboost_cv(tr, features, cat_cols, CAT_PARAMS, n_folds=N_FOLDS, seed=RANDOM_STATE)
pd.DataFrame({'id': tr['id'], 'label': tr['label'], 'oof_pred': oof}).to_csv(os.path.join(OUT_DIR, "oof_catboost_v2.csv"), index=False, encoding='utf-8')
print("[CatBoost] OOF AUC =", round(float(mean_auc), 6), "| folds:", [round(float(x),6) for x in fold_scores])


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6242304	best: 0.6242304 (0)	total: 39.5ms	remaining: 3m 17s
200:	test: 0.6530792	best: 0.6530792 (200)	total: 8.4s	remaining: 3m 20s
400:	test: 0.6575948	best: 0.6576311 (399)	total: 17.2s	remaining: 3m 16s
600:	test: 0.6586624	best: 0.6587685 (591)	total: 25.4s	remaining: 3m 6s
800:	test: 0.6590146	best: 0.6593925 (664)	total: 33.8s	remaining: 2m 57s
bestTest = 0.6593924761
bestIteration = 664
Shrink model to first 665 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6103827	best: 0.6103827 (0)	total: 49.9ms	remaining: 4m 9s
200:	test: 0.6574970	best: 0.6576429 (190)	total: 8.65s	remaining: 3m 26s
400:	test: 0.6594992	best: 0.6595370 (389)	total: 16.6s	remaining: 3m 10s
600:	test: 0.6602844	best: 0.6603335 (474)	total: 24.6s	remaining: 2m 59s
bestTest = 0.6603334546
bestIteration = 474
Shrink model to first 475 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6113801	best: 0.6113801 (0)	total: 35.8ms	remaining: 2m 59s
200:	test: 0.6560614	best: 0.6560614 (200)	total: 8.32s	remaining: 3m 18s
400:	test: 0.6586905	best: 0.6586905 (400)	total: 16.9s	remaining: 3m 14s
600:	test: 0.6591871	best: 0.6593131 (531)	total: 25.5s	remaining: 3m 6s
800:	test: 0.6596671	best: 0.6597914 (793)	total: 33.5s	remaining: 2m 55s
1000:	test: 0.6589838	best: 0.6601886 (858)	total: 41.9s	remaining: 2m 47s
bestTest = 0.6601885557
bestIteration = 858
Shrink model to first 859 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6342394	best: 0.6342394 (0)	total: 39.3ms	remaining: 3m 16s
200:	test: 0.6668764	best: 0.6669664 (199)	total: 8.59s	remaining: 3m 25s
400:	test: 0.6708656	best: 0.6708863 (395)	total: 16.8s	remaining: 3m 12s
600:	test: 0.6724566	best: 0.6724566 (600)	total: 25.2s	remaining: 3m 4s
800:	test: 0.6708967	best: 0.6724895 (601)	total: 34s	remaining: 2m 58s
bestTest = 0.6724895239
bestIteration = 601
Shrink model to first 602 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6192687	best: 0.6192687 (0)	total: 63.8ms	remaining: 5m 18s
200:	test: 0.6653963	best: 0.6655808 (196)	total: 8.46s	remaining: 3m 22s
400:	test: 0.6674844	best: 0.6676425 (368)	total: 16.4s	remaining: 3m 7s
600:	test: 0.6688759	best: 0.6688826 (590)	total: 24.6s	remaining: 2m 59s
800:	test: 0.6681455	best: 0.6691243 (704)	total: 33.2s	remaining: 2m 53s
1000:	test: 0.6663167	best: 0.6691243 (704)	total: 41.4s	remaining: 2m 45s
bestTest = 0.6691243052
bestIteration = 704
Shrink model to first 705 iterations.
[CatBoost] OOF AUC = 0.664094 | folds: [0.659393, 0.660333, 0.660181, 0.67249, 0.669124]


In [6]:

# 测试集预测（testaa / testab 若存在各自输出）
from catboost import Pool as _Pool
def predict_one(te_csv, v2_stm_csv, fallback_csv, tag):
    if not os.path.exists(te_csv):
        print(f"[INFO] {te_csv} 不存在，跳过 {tag}"); return
    te = pd.read_csv(te_csv)
    te = prepare_main_table(te)
    te = merge_statement_feats(te, v2_stm_csv, fallback_csv)
    X_te = te[features].copy()
    pool_te = _Pool(X_te, cat_features=[X_te.columns.get_loc(c) for c in cat_cols])
    preds = np.mean([m.predict_proba(pool_te)[:,1] for m in models], axis=0)
    out = pd.DataFrame({'id': te['id'], 'label': preds})
    path = os.path.join(OUT_DIR, f"test_{tag}_pred_catboost_v2.csv")
    out.to_csv(path, index=False, encoding='utf-8')
    print("[SAVE]", path)

predict_one("testaa.csv", TESTAA_STM_FEAT, "testaa_statement_feature.csv", "aa")
predict_one("testab.csv", TESTAB_STM_FEAT, "testab_statement_feature.csv", "ab")


[SAVE] outputs_cat_v2\test_aa_pred_catboost_v2.csv
[SAVE] outputs_cat_v2\test_ab_pred_catboost_v2.csv
